# Hyperparameter Sweep (Optuna) - PatchCore

This notebook iterates through all 15 MVTec AD categories, performs Optuna trials to find the best hyperparameters, and exports the final mappings for PatchCore.


In [ ]:
import sys
import os
import ctypes
from pathlib import Path
import json
import warnings

# Suppress noisy deprecation warnings from external libraries
warnings.filterwarnings("ignore", category=FutureWarning, module="timm")
warnings.filterwarnings("ignore", category=FutureWarning, module="anomalib")

# Dynamically find the project root (handling both Colab and local VS Code Jupyter)
current_dir = Path.cwd()
while not (current_dir / "app").exists() and current_dir != current_dir.parent:
    current_dir = current_dir.parent
PROJECT_ROOT = current_dir

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Preload CUDA shared libraries into process memory table
lib_dir = os.path.join(sys.prefix, "lib")
for lib_name in [
    "libcudart.so.12", "libcublas.so.12", "libcublasLt.so.12",
    "libcufft.so.11", "libcurand.so.10", "libcusolver.so.11",
    "libcusparse.so.12", "libcudnn.so.9", "libcupti.so.12", "libnvrtc.so.12"
]:
    lib_p = os.path.join(lib_dir, lib_name)
    if os.path.exists(lib_p):
        try:
            ctypes.CDLL(lib_p, mode=ctypes.RTLD_GLOBAL)
        except Exception:
            pass

DATA_ROOT = str(PROJECT_ROOT / "data/raw/mvtec_ad")

CATEGORIES = [
    "bottle", "cable", "capsule", "carpet", "grid",
    "hazelnut", "leather", "metal_nut", "pill", "screw",
    "tile", "toothbrush", "transistor", "wood", "zipper"
]

from app.pipelines.modelling.patchcore_optuna_study import run_study as run_patchcore_study
import gc
import torch

patchcore_output = PROJECT_ROOT / "data/hyperparameters/patchcore_best.json"
patchcore_output.parent.mkdir(parents=True, exist_ok=True)

if patchcore_output.exists():
    patchcore_results = json.loads(patchcore_output.read_text(encoding="utf-8"))
    print(f"Loaded existing progress. Categories already complete: {list(patchcore_results.keys())}")
else:
    patchcore_results = {}

for category in CATEGORIES:
    if category in patchcore_results:
        continue
        
    print(f"\n{'='*50}")
    print(f"Running Optuna study for: {category}")
    print(f"{'='*50}")
    print("\n--- PatchCore ---")
    
    patchcore_cfg = run_patchcore_study(category_name=category, n_trials=30, data_root=DATA_ROOT)
    patchcore_results[category] = patchcore_cfg
    
    # Save incrementally after each category to prevent progress loss
    patchcore_output.write_text(json.dumps(patchcore_results, indent=2))
    print(f"[INFO] Successfully saved {category} progress to {patchcore_output}")
    
    # Aggressively clear memory after each category
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()

print(f"\nSweep complete. PatchCore hyperparameters saved to {patchcore_output.resolve()}")
